# 🎵 Music Search with CLAP + Pinecone

Build a semantic music search engine that understands natural language queries like:
- "chill lo-fi beats for studying"
- "aggressive metal with fast drums"
- "90s hip hop with jazz samples"

## How it works

1. **CLAP** (Contrastive Language-Audio Pretraining) embeds both text and audio in the same vector space
2. **Pinecone** stores the embeddings and enables fast similarity search
3. Your text query gets embedded → finds nearest audio embeddings → returns matching songs

## Prerequisites

- Google account (you're already here!)
- Pinecone account (free tier): https://www.pinecone.io/

---

## Step 1: Setup Environment

First, let's enable GPU and install the required packages.

**Important:** Go to `Runtime → Change runtime type → T4 GPU` before running this notebook!

In [ ]:
# Install required packages (takes ~1-2 minutes)
!pip install -q datasets transformers torch pinecone-client

print("✅ Packages installed!")

In [ ]:
# Check if GPU is available
import torch

if torch.cuda.is_available():
    device = "cuda"
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("⚠️ No GPU detected. Go to Runtime → Change runtime type → T4 GPU")
    print("   (CPU will work but be slower)")

## Step 2: Enter Your Pinecone API Key

Get your free API key from: https://app.pinecone.io/

1. Sign up / Log in
2. Go to "API Keys" in the sidebar
3. Copy your API key

In [ ]:
from getpass import getpass

PINECONE_API_KEY = getpass("Enter your Pinecone API key: ")
print("✅ API key saved (hidden for security)")

## Step 3: Load the CLAP Model

CLAP (from LAION) is like CLIP but for audio:
- **CLIP**: Text ↔ Image (OpenAI)
- **CLAP**: Text ↔ Audio (LAION, open source)

Both text and audio get embedded into the same 512-dimensional vector space.

In [ ]:
from transformers import ClapModel, ClapProcessor

print("Loading CLAP model (first run downloads ~600MB)...")

model = ClapModel.from_pretrained("laion/clap-htsat-unfused")
processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
model = model.to(device)
model.eval()

print("✅ CLAP model loaded!")

## Step 4: Load MusicCaps Dataset

Google's MusicCaps contains 5,521 music clips with human-written captions:
- 10-second clips from YouTube
- Expert captions describing the music
- Genre/mood labels

Example caption: *"A low quality recording of an acoustic guitar playing a sad melody with a slow tempo."*

In [ ]:
from datasets import load_dataset

print("Loading MusicCaps dataset...")
dataset = load_dataset("google/MusicCaps", split="train")

print(f"✅ Loaded {len(dataset)} tracks")
print(f"\nColumns: {dataset.column_names}")
print(f"\nExample caption:\n'{dataset[0]['caption']}'")

## Step 5: Generate CLAP Embeddings

We'll embed the captions using CLAP's text encoder. Since CLAP puts text and audio in the same space, searching with text will find matching audio.

**Note:** For production, you'd embed the actual audio files. We're using captions here for speed.

In [ ]:
import numpy as np
from tqdm import tqdm

# How many tracks to process (start small, increase later)
LIMIT = 1000  # @param {type:"slider", min:100, max:5521, step:100}

vectors = []

print(f"Generating embeddings for {LIMIT} tracks...\n")

for i in tqdm(range(min(LIMIT, len(dataset)))):
    row = dataset[i]
    caption = row.get("caption", "")

    if not caption:
        continue

    # Embed caption with CLAP
    inputs = processor(text=[caption], return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        embedding = model.get_text_features(**inputs)

    # Normalize (important for cosine similarity)
    embedding = embedding[0].cpu().numpy()
    embedding = embedding / np.linalg.norm(embedding)

    vectors.append({
        "id": f"musiccaps_{i}",
        "values": embedding.tolist(),
        "metadata": {
            "caption": caption[:500],
            "ytid": row.get("ytid", ""),
            "youtube_url": f"https://youtube.com/watch?v={row.get('ytid', '')}&t={int(row.get('start_s', 0))}",
            "labels": row.get("audioset_positive_labels", ""),
        }
    })

print(f"\n✅ Generated {len(vectors)} embeddings")
print(f"   Dimensions: {len(vectors[0]['values'])}")

## Step 6: Create Pinecone Index & Upload

Now we'll store these embeddings in Pinecone for fast similarity search.

In [ ]:
from pinecone import Pinecone, ServerlessSpec

INDEX_NAME = "music-search"

# Initialize Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

# Check if index exists
existing_indexes = [idx.name for idx in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    print(f"Creating index '{INDEX_NAME}'...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=512,  # CLAP embedding size
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Waiting for index to be ready...")
    import time
    time.sleep(30)
else:
    print(f"Index '{INDEX_NAME}' already exists")

index = pc.Index(INDEX_NAME)
print(f"✅ Connected to index")

In [ ]:
# Upload embeddings in batches
BATCH_SIZE = 100

print(f"Uploading {len(vectors)} vectors...")

for i in tqdm(range(0, len(vectors), BATCH_SIZE)):
    batch = vectors[i:i+BATCH_SIZE]
    index.upsert(vectors=batch)

print(f"\n✅ Uploaded {len(vectors)} vectors!")

# Verify
stats = index.describe_index_stats()
print(f"   Total vectors in index: {stats.total_vector_count}")

## Step 7: Search! 🎵

Now the fun part - search for music using natural language!

In [ ]:
def search_music(query: str, top_k: int = 5):
    """Search for music using a text description."""

    # Embed the query with CLAP
    inputs = processor(text=[query], return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        query_embedding = model.get_text_features(**inputs)

    # Normalize
    query_embedding = query_embedding[0].cpu().numpy()
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    # Search Pinecone
    results = index.query(
        vector=query_embedding.tolist(),
        top_k=top_k,
        include_metadata=True
    )

    # Display results
    print("=" * 70)
    print(f"🔍 Results for: \"{query}\"")
    print("=" * 70)

    for i, match in enumerate(results.matches, 1):
        meta = match.metadata or {}
        score = match.score

        print(f"\n{i}. Score: {score:.4f}")
        print(f"   📝 {meta.get('caption', 'No caption')[:100]}...")
        if meta.get('youtube_url'):
            print(f"   🎬 {meta['youtube_url']}")

    print("\n" + "=" * 70)
    return results

In [ ]:
# Try some searches!
search_music("chill lo-fi beats for studying")

In [ ]:
search_music("aggressive metal with fast drums")

In [ ]:
search_music("upbeat electronic dance music")

In [ ]:
search_music("sad piano ballad")

In [ ]:
# 🎯 Try your own search!
YOUR_QUERY = "jazz with saxophone"  # @param {type:"string"}

search_music(YOUR_QUERY)

---

## 🎉 Congratulations!

You've built a semantic music search engine using:

1. **CLAP** - Open source model that embeds text and audio in the same space
2. **MusicCaps** - Google's dataset of music with captions
3. **Pinecone** - Vector database for fast similarity search

### Key Concepts

- **Embeddings**: Convert text/audio into numerical vectors that capture meaning
- **Vector similarity**: Similar meanings → vectors close together
- **Cross-modal search**: Search audio using text (or vice versa)

### Next Steps

- Increase `LIMIT` to index all 5,521 tracks
- Try embedding actual audio files (not just captions)
- Build a web UI with Gradio or Streamlit
- Compare with other embedding models (Whisper, MusicGen)

### Resources

- [CLAP Paper](https://arxiv.org/abs/2211.06687)
- [LAION CLAP on Hugging Face](https://huggingface.co/laion/clap-htsat-unfused)
- [MusicCaps Dataset](https://huggingface.co/datasets/google/MusicCaps)
- [Pinecone Docs](https://docs.pinecone.io/)